# Fine-tune LLM on Amazon SageMaker AI with Quota Management Queue

This notebook demonstrates how to submit training jobs to an AWS Batch **quota management** queue.
Quota management provides explicit capacity allocation via **quota shares**, with support for:
- **Resource sharing** — lending and borrowing idle capacity between shares
- **Cross-share preemption** — automatically reclaiming lent capacity when needed
- **In-share preemption** — higher-priority jobs preempting lower-priority ones within the same share
- **Dynamic priority updates** — changing job priority after submission

This notebook reuses the same model fine-tuning setup as the fair-share notebook, but targets a `quota-management` queue instead.

## Prerequisites

In [ ]:
! pip install -r ./scripts/requirements.txt --upgrade

***

## Setup Configuration

In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join(".."))

if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import os

model_id = "Qwen/Qwen3-0.6B"

os.environ["model_id"] = model_id

***

## Visualize and upload the dataset

We are going to load [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset

In [ ]:
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[:1000]")

df = pd.DataFrame(dataset)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.1, random_state=42)

print("Number of train elements: ", len(train))
print("Number of test elements: ", len(val))

Create a prompt template and load the dataset with a random sample to try summarization.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)

def prepare_dataset(sample):
    system_text = (
        "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.\n"
        "Below is an instruction that describes a task, paired with an input that provides further context.\n"
        "Write a response that appropriately completes the request.\n"
        "Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response."
    )

    messages = []
    messages.append({"role": "system", "content": system_text})
    messages.append({"role": "user", "content": sample["Question"]})

    # Use different tags that won't be detected by the template
    messages.append(
        {
            "role": "assistant",
            "content": f"\n[REASONING_START]\n{sample['Complex_CoT']}\n[REASONING_END]\n{sample['Response']}",
        }
    )

    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)

    # Replace with actual think tags after template processing
    formatted_text = formatted_text.replace("[REASONING_START]", "<think>")
    formatted_text = formatted_text.replace("[REASONING_END]", "</think>")

    sample["text"] = formatted_text

    return sample

Use the Hugging Face Trainer class to fine-tune the model. Define the hyperparameters we want to use. We also create a DataCollator that will take care of padding our inputs and labels.

In [ ]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)

dataset = DatasetDict({"train": train_dataset, "val": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset, remove_columns=list(train_dataset.features)
)

print(train_dataset[randint(0, len(dataset))]["text"])

val_dataset = dataset["val"].map(
    prepare_dataset, remove_columns=list(val_dataset.features)
)

### Upload to Amazon S3

In [ ]:
import boto3
import shutil
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
s3_client = boto3.client('s3')

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-sft-batch"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-sft-batch"

# Save datasets to s3
train_dataset.to_json("./data/train/dataset.json", orient="records")
val_dataset.to_json("./data/val/dataset.json", orient="records")

s3_client.upload_file("./data/train/dataset.json", bucket_name, f"{input_path}/train/dataset.json")
train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.json"
s3_client.upload_file("./data/val/dataset.json", bucket_name, f"{input_path}/val/dataset.json")
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.json"

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)

***

## Model fine-tuning

We are now ready to fine-tune our model. We will use the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) from transfomers to fine-tune our model. We prepared a script [train.py](./scripts/train.py) which will loads the dataset from disk, prepare the model, tokenizer and start the training.

For configuration we use `TrlParser`, that allows us to provide hyperparameters in a `yaml` file. This yaml will be uploaded and provided to Amazon SageMaker similar to our datasets. Below is the config file for fine-tuning the model on `ml.g5.12xlarge`. We are saving the config file as `args.yaml` and upload it to S3.

In [ ]:
%%bash

cat > ./args.yaml <<EOF
model_id: "${model_id}"       # Hugging Face model id
# sagemaker specific parameters
output_dir: "/opt/ml/model"                       # path to where SageMaker will upload the model 
checkpoint_dir: "/opt/ml/checkpoints/"            # directory for saving training checkpoints
train_dataset_path: "/opt/ml/input/data/train/"   # path to where S3 saves train dataset
val_dataset_path: "/opt/ml/input/data/val/"       # path to where S3 saves test dataset
# training parameters
apply_truncation: true                           # apply truncation to datasets
attn_implementation: "flash_attention_2"         # attention implementation type
learning_rate: 2.80e-04                          # learning rate scheduler
num_train_epochs: 2                              # number of training epochs
per_device_train_batch_size: 2                   # batch size per device during training
per_device_eval_batch_size: 1                    # batch size for evaluation
gradient_accumulation_steps: 2                   # number of steps before performing a backward/update pass
gradient_checkpointing: true                     # use gradient checkpointing
torch_dtype: "bfloat16"                          # float precision type
bf16: true                                       # use bfloat16 precision
tf32: true                                       # use tf32 precision
ignore_data_skip: true                           # skip data loading errors
logging_strategy: "steps"                        # logging strategy
logging_steps: 1                                 # log every N steps
log_on_each_node: false                          # disable logging on each node
ddp_find_unused_parameters: false                # DDP unused parameter detection
save_total_limit: 1                              # maximum number of checkpoints to keep
save_steps: 90                                   # Save checkpoint every this many steps
warmup_steps: 27                                 # number of warmup steps
weight_decay: 5.00e-02                           # weight decay coefficient
fsdp: "full_shard auto_wrap offload"             # FSDP sharding strategy
fsdp_config:                                     # FSDP configuration options
    backward_prefetch: "backward_pre"            # prefetch parameters during backward pass
    cpu_ram_efficient_loading: true              # enable CPU RAM efficient model loading
    offload_params: false                        # offload parameters to CPU
    forward_prefetch: false                      # disable forward prefetch
    use_orig_params: true                        # use original parameter names
# LoRA parameters
load_in_4bit: true                               # enable 4-bit quantization
lora_r: 16                                       # LoRA rank
lora_alpha: 32                                   # LoRA alpha parameter
lora_dropout: 0.1                                # LoRA dropout rate
EOF

Lets upload the config file to S3.

In [ ]:
import os
from sagemaker.core.s3 import S3Uploader

if default_prefix:
    input_path = f"s3://{bucket_name}/{default_prefix}/datasets/llm-fine-tuning-modeltrainer-sft-batch"
else:
    input_path = f"s3://{bucket_name}/datasets/llm-fine-tuning-modeltrainer-sft-batch"

# upload the model yaml file to s3
model_yaml = "args.yaml"
train_config_s3_path = S3Uploader.upload(local_path=model_yaml, desired_s3_uri=f"{input_path}/config")

os.remove("./args.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

## Create the ModelTrainer

Below the ModelTrainer will be used to submit the jobs

#### Get PyTorch image_uri

We are going to use the native PyTorch container image, pre-built for Amazon SageMaker

In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
instance_type = "ml.g5.xlarge"
instance_count = 1

instance_type

In [ ]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

In [ ]:
from sagemaker.train.configs import Compute, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train.distributed import Torchrun
from sagemaker.train.model_trainer import ModelTrainer

role = get_execution_role()

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=1,
    keep_alive_period_in_seconds=0
)

# define Training Job Name
job_name = f"train-{model_id.split('/')[-1].replace('.', '-')}-qm"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={
        "config": "/opt/ml/input/data/config/args.yaml"  # path to TRL config which was uploaded to s3
    },
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    role=role,
)

In [ ]:
from sagemaker.train.configs import InputData

# Pass the input data
train_input = InputData(
    channel_name="train",
    data_source=train_dataset_s3_path, # S3 path where training data is stored
)

val_input = InputData(
    channel_name="val",
    data_source=val_dataset_s3_path,  # S3 path where training data is stored
)

config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path, # S3 path where training data is stored
)

# Check input channels configured
data = [train_input, val_input, config_input]

data

***

## Connect to the Quota Management Queue

We use the `TrainingQueue` class from the SageMaker Python SDK to interact with our quota management queue.
The queue `team-c-queue` is configured with three quota shares:
- **QS1** — `LEND_AND_BORROW` (borrow limit 200%), in-share preemption enabled
- **QS2** — `LEND` (lends idle capacity, no borrowing)
- **QS3** — `RESERVE` (capacity exclusively reserved)

In [ ]:
from sagemaker.train.aws_batch.training_queue import TrainingQueue

SMTJ_BATCH_QUEUE = "team-c-queue"

queue = TrainingQueue(SMTJ_BATCH_QUEUE)
print(f"Using queue: {queue.queue_name}")

## Step 1: Submit three jobs to QS1 with different priorities

QS1 has capacity for 1 instance, but can borrow up to 200% from other shares.
We submit 3 jobs with priorities 3 (high), 2 (med), and 1 (low):
- `qs1_job_high` (priority=3) dispatches using QS1's own capacity
- `qs1_job_med` (priority=2) borrows idle capacity from QS2 (which is configured as LEND)
- `qs1_job_low` (priority=1) stays RUNNABLE — QS3 is RESERVE, no more capacity to borrow

In [ ]:
qs1_job_high = queue.submit(
    model_trainer, data,
    job_name=job_name + "-qs1-high",
    quota_share_name="QS1",
    priority=3,
)

qs1_job_med = queue.submit(
    model_trainer, data,
    job_name=job_name + "-qs1-med",
    quota_share_name="QS1",
    priority=2,
)

qs1_job_low = queue.submit(
    model_trainer, data,
    job_name=job_name + "-qs1-low",
    quota_share_name="QS1",
    priority=1,
)

print(f"Submitted: {qs1_job_high.job_name} (pri=3)\n{qs1_job_med.job_name} (pri=2)\n{qs1_job_low.job_name} (pri=1)")

### Check queue state

Wait a moment for the scheduler to dispatch, then check the state. Expected:
- `qs1_job_high` — STARTING/RUNNING (on QS1's own capacity)
- `qs1_job_med` — STARTING/RUNNING (borrowing from QS2)
- `qs1_job_low` — RUNNABLE (no capacity available to borrow)

In [ ]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

## Step 2: Submit a job to QS2 — triggers cross-share preemption

When a job arrives for QS2, it needs its own capacity back. QS1 was borrowing QS2's slot
to run `qs1_job_med`, so **cross-share preemption** kicks in:
- `qs1_job_med` gets preempted (goes back to RUNNABLE)
- `qs2_job_1` takes QS2's slot

In [ ]:
qs2_job_1 = queue.submit(
    model_trainer, data,
    job_name=job_name + "-qs2-job1",
    quota_share_name="QS2",
    priority=1,
)

print(f"Submitted: {qs2_job_1.job_name} to QS2 — this should trigger cross-share preemption")

### Check queue state after cross-share preemption

Expected:
- `qs1_job_high` — RUNNING (on QS1's own capacity, unaffected)
- `qs2_job_1` — STARTING/RUNNING (on QS2's reclaimed capacity)
- `qs1_job_med` — RUNNABLE (preempted, waiting for capacity)
- `qs1_job_low` — RUNNABLE

In [ ]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

## Step 3: Update job priority — triggers in-share preemption

QS1 has in-share preemption enabled. We can update `qs1_job_low`'s priority to 4
(higher than `qs1_job_high`'s priority of 3). This triggers **in-share preemption**:
- `qs1_job_high` (priority=3) gets preempted
- `qs1_job_low` (now priority=4) takes QS1's slot

This demonstrates **dynamic priority updates** — a capability unique to quota management queues.

In [ ]:
# Promote qs1_job_low to highest priority in QS1
qs1_job_low.update(scheduling_priority=4)

print(f"Updated {qs1_job_low.job_name} priority to 4 (was 1)")
print("This should trigger in-share preemption of qs1_job_high (priority=3)")

### Check queue state after in-share preemption

Expected:
- `qs1_job_low` — STARTING/RUNNING (promoted to priority=4, took QS1's slot)
- `qs2_job_1` — RUNNING (unaffected)
- `qs1_job_high` — RUNNABLE (preempted by the now-higher-priority qs1_job_low)
- `qs1_job_med` — RUNNABLE

In [ ]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

## Check preemption history

The `describe()` response includes a `preemptionSummary` field that shows how many times a job has been preempted and the details of recent preemption attempts.

In [ ]:
import json

for job, label in [(qs1_job_high, "qs1_job_high"), (qs1_job_med, "qs1_job_med"), (qs1_job_low, "qs1_job_low"), (qs2_job_1, "qs2_job_1")]:
    desc = job.describe()
    status = desc.get("status", "")
    preemption = desc.get("preemptionSummary", {})
    preempted_count = preemption.get("preemptedAttemptCount", 0)
    print(f"{label}: status={status}, preempted {preempted_count} time(s)")
    if preempted_count > 0:
        for attempt in preemption.get("recentPreemptedAttempts", []):
            print(f"  - reason: {attempt.get('statusReason', 'N/A')}")

## List jobs by quota share

The SDK supports filtering jobs by quota share name using `list_jobs_by_share()`.

In [ ]:
for qs_name in ["QS1", "QS2", "QS3"]:
    print(f"\n{qs_name}:")
    found = False
    for status in ["SUBMITTED", "PENDING", "RUNNABLE", "STARTING", "RUNNING", "SUCCEEDED", "FAILED"]:
        jobs = queue.list_jobs_by_share(quota_share_name=qs_name, status=status)
        for j in jobs:
            found = True
            print(f"  {j.job_name}: {status}")
    if not found:
        print("  (no jobs)")

***

## Cancel remaining jobs

Clean up by terminating all runnable and running jobs.

In [ ]:
for status in ["RUNNABLE", "STARTING", "RUNNING"]:
    jobs = queue.list_jobs(status=status)
    for job in jobs:
        print(f"Terminating: {job.job_name} ({status})")
        job.terminate(reason="Notebook cleanup")

print("\nAll jobs terminated.")